# The Complete Introduction to Proximal Policy Optimization

Welcome to state-of-the-art Reinforcement Learning. Proximal Policy Optimization (PPO) is an **Actor-Critic** method. 

1. **The Actor (Policy):** Looks at the environment state and decides what action to take. It outputs a probability distribution over actions.
2. **The Critic (Value Function):** Looks at the environment state and guesses how much total reward the agent will get from this point onward. 

PPO trains both networks simultaneously. The Critic calculates the **Advantage** (did the action turn out better or worse than we expected?), and the Actor updates its probabilities based on that Advantage—but uses the clipping function to ensure it doesn't take too large of a step.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing `gymnasium` and `stable-baselines3`.
2. **Environment Analysis:** Understanding the State and Action spaces.
3. **Model Instantiation:** Building the PPO Actor-Critic architecture.
4. **Training Loop:** Letting the agent learn through trial, error, and clipped gradients.
5. **Evaluation:** Testing the trained agent mathematically.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, you must install: !pip install gymnasium stable-baselines3[extra]

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully. Ready for Reinforcement Learning.")

## Step 1: Initializing the Environment

Reinforcement Learning requires an environment formulated as a Markov Decision Process (MDP). We will use the classic **CartPole-v1** environment.

**The Goal:** Balance a pole on a moving cart.
* **Reward:** +1 for every timestep the pole remains upright.
* **Termination:** The episode ends if the pole falls over (angle > 12 degrees) or the cart moves off screen.

Before training, an ML engineer must always analyze the mathematical bounds of the **Observation Space** (what the AI sees) and the **Action Space** (what the AI can do).

In [ ]:
# Cell 4: Environment Analysis

# Create the environment
env = gym.make("CartPole-v1")

# Reset the environment to get the initial state
obs, info = env.reset(seed=42)

print("--- Environment Analysis ---")
print(f"Observation Space: {env.observation_space}")
# The observation is a 4D continuous vector: [Cart Position, Cart Velocity, Pole Angle, Pole Angular Velocity]
print(f"Initial State Vector: {obs}")

print(f"\nAction Space: {env.action_space}")
# The action is Discrete(2): 0 means push cart Left, 1 means push cart Right
print(f"Sample Random Action: {env.action_space.sample()}")

## Step 2: Defining the PPO Architecture

Because we are using `stable-baselines3`, creating the complex Actor-Critic neural network architecture is abstracted into a single line. 

We will use the `MlpPolicy`, which builds standard Multi-Layer Perceptrons for both the Actor network and the Critic network. 

**Analytical Hyperparameters:**
* `learning_rate`: How fast the neural networks update their weights.
* `n_steps`: The number of steps to run in the environment before performing a mathematical update. (PPO collects a batch of experiences, then updates).
* `clip_range`: The $\epsilon$ parameter from our widget above. Usually set to `0.2`.

In [ ]:
# Cell 6: Model Instantiation

# We instantiate the PPO algorithm
model = PPO(
    policy="MlpPolicy", 
    env=env, 
    learning_rate=0.0003, 
    n_steps=2048,           # Collect 2048 timesteps before updating
    batch_size=64,          # Mini-batch size for the gradient descent
    n_epochs=10,            # Number of times to optimize over the batch
    clip_range=0.2,         # The crucial PPO clipping parameter
    gamma=0.99,             # Discount factor (how much it cares about future rewards vs immediate rewards)
    verbose=0               # Set to 1 to see training logs
)

print("PPO Actor-Critic model successfully initialized.")

## Step 3: Training the Agent

In standard Supervised Learning, you pass a dataset `X` and labels `y` into `model.fit()`. 
In Reinforcement Learning, there is no dataset! The agent must generate its own data by interacting with the environment.

When we call `model.learn()`, PPO begins the loop:
1. Play the game using the current policy for `n_steps`.
2. Calculate the Advantages using the Critic.
3. Calculate the Clipped Objective.
4. Perform Gradient Ascent to maximize the objective (updating the neural network weights).
5. Repeat.

In [ ]:
# Cell 8: Executing the Training Loop

# Let's test the agent BEFORE training to establish a baseline
mean_reward_before, std_reward_before = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Untrained Agent - Mean Reward: {mean_reward_before:.2f} +/- {std_reward_before:.2f}")
# (A perfectly random agent usually scores around 15 to 25 before dropping the pole)

print("\nStarting Training (Simulating 20,000 timesteps)...")

# Train the model
model.learn(total_timesteps=20_000)

print("Training Complete!")

## Step 4: Analytical Evaluation

How do we know the clipping and policy gradients actually worked? We evaluate the trained policy.

Instead of acting randomly, the agent will now ask its trained Actor network for the best action given the current state. A successfully trained CartPole agent should achieve the maximum score of `500.0` perfectly, meaning it balanced the pole until the environment's hard time limit ran out.

In [ ]:
# Cell 10: Evaluating the Trained Agent

# Evaluate the trained agent over 10 independent episodes
mean_reward_after, std_reward_after = evaluate_policy(model, env, n_eval_episodes=10)

print("--- Final Evaluation ---")
print(f"Trained Agent - Mean Reward: {mean_reward_after:.2f} +/- {std_reward_after:.2f}")

if mean_reward_after == 500.0:
    print("\nAnalytical Conclusion: The PPO algorithm successfully converged to the optimal policy.")
    print("The agent learned to perfectly balance the pole by taking small, controlled, clipped steps through the mathematical loss landscape!")
else:
    print("\nAnalytical Conclusion: The agent improved, but may need more timesteps to reach perfection.")

# Close the environment to free up memory
env.close()